# 🔍 CrackSense – UNet Crack Segmentor
**Dataset:** [CrackVision12K](https://www.kaggle.com/datasets/vangiap/crackvision12k)  
**Goal:** Pixel-level crack segmentation + heat-map overlay on original image  
**Model:** U-Net (encoder–bottleneck–decoder with skip connections)  
**Loss:** Binary Cross-Entropy + Dice Loss  

---

## 1. Setup & Imports

In [ ]:
import os, sys, warnings, json, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image as PILImage

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import (
    KAGGLE_DATASET_HANDLE, UNET_IMG_SIZE, UNET_BATCH_SIZE,
    UNET_EPOCHS, LEARNING_RATE, VALIDATION_SPLIT,
    UNET_MODEL_PATH, SAVED_MODELS_DIR,
    OVERLAY_ALPHA, OVERLAY_COLOR
)
from src.models.unet_model import build_unet_segmentor
from src.utils.image_utils import generate_overlay, compute_crack_coverage

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}')

## 2. Download Dataset via Kaggle Hub

In [ ]:
import kagglehub

print(f'Downloading: {KAGGLE_DATASET_HANDLE} ...')
dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET_HANDLE))
print(f'Cached at: {dataset_path}')

## 3. Build Pseudo-Masks from Classification Images
This historical workflow assumes class-folder images and generates pseudo-masks using adaptive thresholding and morphology. Negative images receive zero masks. This implementation choice does not establish whether all dataset releases lack annotated masks. Prefer verified pixel annotations for supervised segmentation evaluation.


In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

# Locate class folders
for d in dataset_path.rglob('*'):
    if d.is_dir():
        children = [c for c in d.iterdir() if c.is_dir()]
        if len(children) == 2:
            data_root = d
            class_dirs = {c.name.lower(): c for c in children}
            break

print(f'Data root : {data_root}')
print(f'Classes   : {list(class_dirs.keys())}')

# Identify positive (crack) class
pos_key = next((k for k in class_dirs if any(
    w in k for w in ['positive','crack','yes'])), list(class_dirs.keys())[0])
neg_key = [k for k in class_dirs if k != pos_key][0]

pos_imgs = [f for f in class_dirs[pos_key].rglob('*') if f.suffix.lower() in IMG_EXTS]
neg_imgs = [f for f in class_dirs[neg_key].rglob('*') if f.suffix.lower() in IMG_EXTS]

print(f'Positive (crack): {len(pos_imgs):,}')
print(f'Negative (no-crack): {len(neg_imgs):,}')

In [ ]:
def generate_pseudo_mask(img_bgr: np.ndarray) -> np.ndarray:
    """
    Generate a binary crack mask using adaptive thresholding.
    Returns float32 mask in [0,1], same H×W as input.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    # CLAHE for contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    # Adaptive threshold
    thresh = cv2.adaptiveThreshold(
        enhanced, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        blockSize=15, C=5
    )
    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    mask = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=1)
    return mask.astype(np.float32) / 255.0

# Preview pseudo-masks on 4 positive images
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
sample_pos = random.sample(pos_imgs, 4)
for i, img_path in enumerate(sample_pos):
    img_bgr = cv2.imread(str(img_path))
    mask    = generate_pseudo_mask(img_bgr)
    overlay = generate_overlay(img_bgr, mask)

    axes[0, i].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title('Original', fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(mask, cmap='hot')
    axes[1, i].set_title('Pseudo-Mask', fontsize=9)
    axes[1, i].axis('off')

    axes[2, i].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[2, i].set_title('Overlay', fontsize=9)
    axes[2, i].axis('off')

plt.suptitle('Pseudo-Mask Generation Preview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'saved_models' / 'unet_pseudo_masks_preview.png'), dpi=150)
plt.show()

## 4. Build tf.data Pipeline

In [ ]:
IMG_H, IMG_W = UNET_IMG_SIZE

# Build file-label list: (path, is_positive)
all_paths = ([(str(p), 1) for p in pos_imgs] +
             [(str(p), 0) for p in neg_imgs])
random.shuffle(all_paths)

# Use a manageable subset if dataset is large (optional speedup)
MAX_SAMPLES = 6000  # increase for full training
if len(all_paths) > MAX_SAMPLES:
    pos_subset = [(p, l) for p, l in all_paths if l == 1][:MAX_SAMPLES//2]
    neg_subset = [(p, l) for p, l in all_paths if l == 0][:MAX_SAMPLES//2]
    all_paths  = pos_subset + neg_subset
    random.shuffle(all_paths)

split_idx = int(len(all_paths) * (1 - VALIDATION_SPLIT))
train_paths = all_paths[:split_idx]
val_paths   = all_paths[split_idx:]

print(f'Train: {len(train_paths)} | Val: {len(val_paths)}')

def load_sample(path, is_positive):
    """Load image + generate mask on-the-fly."""
    path_str = path.numpy().decode('utf-8')
    is_pos   = bool(is_positive.numpy())

    img_bgr = cv2.imread(path_str)
    if img_bgr is None:
        img_bgr = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)

    img_bgr = cv2.resize(img_bgr, (IMG_W, IMG_H))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_f   = img_rgb.astype(np.float32) / 255.0

    if is_pos:
        mask = generate_pseudo_mask(img_bgr)
    else:
        mask = np.zeros((IMG_H, IMG_W), dtype=np.float32)

    mask = cv2.resize(mask, (IMG_W, IMG_H))
    mask = mask[..., np.newaxis]  # H,W,1
    return img_f, mask

def tf_load(path, label):
    img, mask = tf.py_function(load_sample, [path, label],
                               [tf.float32, tf.float32])
    img.set_shape( [IMG_H, IMG_W, 3])
    mask.set_shape([IMG_H, IMG_W, 1])
    return img, mask

@tf.function
def augment(img, mask):
    """Random flip + brightness/contrast."""
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, mask

def make_dataset(path_label_list, augment_flag=False, batch=UNET_BATCH_SIZE):
    paths  = [p for p, _ in path_label_list]
    labels = [l for _, l in path_label_list]
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths), tf.constant(labels, dtype=tf.int32))
    )
    ds = ds.map(tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_flag:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, augment_flag=True)
val_ds   = make_dataset(val_paths,   augment_flag=False)

print('Datasets ready.')

## 5. Build UNet + Custom Dice-BCE Loss

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    dice = (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )
    return 1.0 - dice

def bce_dice_loss(y_true, y_pred):
    bce  = keras.losses.binary_crossentropy(y_true, y_pred)
    d    = dice_loss(y_true, y_pred)
    return 0.5 * bce + 0.5 * d

def iou_metric(y_true, y_pred, threshold=0.5):
    y_pred_b = tf.cast(y_pred > threshold, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred_b)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred_b) - intersection
    return (intersection + 1e-6) / (union + 1e-6)

def dice_coeff(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )

unet = build_unet_segmentor()
unet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=bce_dice_loss,
    metrics=[iou_metric, dice_coeff, 'binary_accuracy']
)
unet.summary()
print(f'\nTotal params: {unet.count_params():,}')

## 6. Train UNet

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_iou_metric', patience=10,
        restore_best_weights=True, mode='max', verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-8, verbose=1
    ),
    ModelCheckpoint(
        filepath=UNET_MODEL_PATH,
        monitor='val_iou_metric', save_best_only=True,
        mode='max', verbose=1
    ),
]

history = unet.fit(
    train_ds,
    epochs=UNET_EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1,
)

print(f'\n✅ Best UNet saved to: {UNET_MODEL_PATH}')

## 7. Training Curves

In [ ]:
hist = history.history
epochs_range = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
pairs = [
    ('loss', 'val_loss', 'BCE + Dice Loss', '#ef4444'),
    ('iou_metric', 'val_iou_metric', 'IoU Score', '#3b82f6'),
    ('dice_coeff', 'val_dice_coeff', 'Dice Coefficient', '#22c55e'),
]
for ax, (tr, vl, title, color) in zip(axes, pairs):
    ax.plot(epochs_range, hist[tr],  label='Train', color=color, linewidth=2)
    ax.plot(epochs_range, hist[vl],  label='Val',   color=color, linewidth=2, linestyle='--')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('UNet Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'saved_models' / 'unet_training_curves.png'), dpi=150)
plt.show()

## 8. Qualitative Results – Overlays

In [ ]:
# Visualise predictions on 6 validation samples
val_sample_paths = random.sample([p for p, l in val_paths if l == 1], min(6, len(val_paths)))

fig, axes = plt.subplots(3, len(val_sample_paths), figsize=(18, 10))

total_iou, total_dice = [], []

for i, img_path in enumerate(val_sample_paths):
    img_bgr  = cv2.imread(img_path)
    gt_mask  = generate_pseudo_mask(img_bgr)

    resized  = cv2.resize(img_bgr, (IMG_W, IMG_H))
    tensor   = resized.astype(np.float32) / 255.0
    tensor   = cv2.cvtColor(tensor, cv2.COLOR_BGR2RGB)
    pred     = unet.predict(tensor[np.newaxis], verbose=0)[0, :, :, 0]

    overlay  = generate_overlay(img_bgr, pred)

    # IoU / Dice vs pseudo-GT
    gt_r   = cv2.resize(gt_mask, (IMG_W, IMG_H))
    p_bin  = (pred > 0.5).astype(np.float32)
    inter  = np.sum(gt_r * p_bin)
    union  = np.sum(gt_r) + np.sum(p_bin) - inter
    iou    = inter / (union + 1e-6)
    dice   = 2 * inter / (np.sum(gt_r) + np.sum(p_bin) + 1e-6)
    total_iou.append(iou); total_dice.append(dice)

    axes[0, i].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title('Original', fontsize=8); axes[0, i].axis('off')

    axes[1, i].imshow(pred, cmap='hot')
    axes[1, i].set_title(f'Pred mask\nIoU={iou:.3f}', fontsize=8); axes[1, i].axis('off')

    axes[2, i].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    cov = compute_crack_coverage(pred)
    axes[2, i].set_title(f'Overlay\nCoverage: {cov}%', fontsize=8); axes[2, i].axis('off')

print(f'Mean IoU  : {np.mean(total_iou):.4f}')
print(f'Mean Dice : {np.mean(total_dice):.4f}')

plt.suptitle('UNet Predictions – Original | Mask | Overlay', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'saved_models' / 'unet_qualitative_results.png'), dpi=150)
plt.show()

## 9. Save Metadata

In [ ]:
assert os.path.exists(UNET_MODEL_PATH), '❌ UNet model file missing!'
size_mb = os.path.getsize(UNET_MODEL_PATH) / (1024 * 1024)
print(f'✅ UNet saved: {UNET_MODEL_PATH}  ({size_mb:.2f} MB)')

meta = {
    'model': 'CrackUNet',
    'dataset': KAGGLE_DATASET_HANDLE,
    'input_size': list(UNET_IMG_SIZE),
    'loss': 'BCE + Dice',
    'best_val_iou':  float(max(hist['val_iou_metric'])),
    'best_val_dice': float(max(hist['val_dice_coeff'])),
    'mean_iou_qualitative':  float(np.mean(total_iou)),
    'mean_dice_qualitative': float(np.mean(total_dice)),
    'overlay_alpha': OVERLAY_ALPHA,
    'overlay_color_bgr': list(OVERLAY_COLOR),
}
with open(str(PROJECT_ROOT / 'saved_models' / 'unet_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('\n📋 Metadata:')
print(json.dumps(meta, indent=2))